# Figure 2 — Clustering-method benchmark

Run top-to-bottom to replay the figure from the verified H5AD, exact source-traced clustering assignments, and model API responses.


## Publication output preview

![Figure 2 publication output](previews/figure_02.png)

This source-traced preview is from the validated publication run. Code-cell outputs remain unexecuted in version control. The analytical replay uses the public H5AD, tracked frozen assignments, and raw or live LLM responses; rerunning the upstream TIFF-based clustering engines is a separate workflow.


## 1. Configure the source data and model inputs


In [ ]:
import os
import sys
from pathlib import Path
from IPython.display import display


def _canonical_repository_root(candidate: Path) -> Path | None:
    direct_package = candidate / "src" / "llm_spatial_omics_clustering"
    if direct_package.is_dir() and (candidate / "notebooks" / "final_figures").is_dir():
        return candidate
    nested = candidate / "LLM-Spatial-omics-Clustering"
    nested_package = nested / "src" / "llm_spatial_omics_clustering"
    if nested_package.is_dir() and (nested / "notebooks" / "final_figures").is_dir():
        return nested
    return None


def find_repository_root() -> Path:
    configured = os.environ.get("SOURCE_REBUILD_REPO_ROOT")
    if configured:
        root = _canonical_repository_root(Path(configured).expanduser().resolve())
        if root is None:
            raise RuntimeError(
                "SOURCE_REBUILD_REPO_ROOT must be this repository or its parent directory"
            )
        return root
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        root = _canonical_repository_root(candidate)
        if root is not None:
            return root
    raise RuntimeError("Run from this repository checkout or set SOURCE_REBUILD_REPO_ROOT")


REPO_ROOT = find_repository_root()

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from llm_spatial_omics_clustering.duke_h5ad import download_duke_h5ad
DUKE_H5AD_PATH = download_duke_h5ad(REPO_ROOT / "data" / "raw" / "duke_research_repository").path

# The analytical replay uses the public H5AD, tracked frozen assignments, and raw/live LLM responses.
# Edit paths here or set the same variables before launching Jupyter.
os.environ.setdefault(
    "SOURCE_REBUILD_H5AD_PATH",
    str(DUKE_H5AD_PATH),
)
os.environ.setdefault("SOURCE_REBUILD_ASSIGNMENTS_ROOT", str(REPO_ROOT / "data" / "frozen" / "v3_k300_assignments"))
# TIFF assets are needed only to rerun the upstream image-native PIXIE engine,
# not to replay the source-traced publication assignments tracked here.
os.environ.setdefault("SOURCE_REBUILD_LLM_MODE", "live")
os.environ.setdefault(
    "SOURCE_REBUILD_LLM_OUTPUT_ROOT", str(REPO_ROOT / "outputs" / "source_rebuilt" / "raw_llm")
)
os.environ.setdefault("SOURCE_REBUILD_OUTPUT_ROOT", str(REPO_ROOT / "outputs" / "source_rebuilt"))

# Leave this literal placeholder in the notebook. Supply the real key through the shell/Jupyter environment.
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "REPLACE_WITH_OPENROUTER_API_KEY")
if os.environ["SOURCE_REBUILD_LLM_MODE"].strip().lower() == "live" and OPENROUTER_API_KEY == "REPLACE_WITH_OPENROUTER_API_KEY":
    print("Live LLM annotation requires OPENROUTER_API_KEY; frozen assignments can still be validated first.")


## 2. Validate the frozen publication assignments

The four tracked gzip files are byte-preserving copies of the exact V3
assignments used in the source-traced publication output. Every method was
configured with K=300; Leiden, FlowSOM, and PIXIE occupy 300 labels, while
SpatialSort occupies 246. The runtime verifies both compressed and
decompressed hashes, 220,082 exact cell keys, and occupied-cluster counts.


In [ ]:
import json

import pandas as pd

FROZEN_ASSIGNMENT_ROOT = Path(os.environ["SOURCE_REBUILD_ASSIGNMENTS_ROOT"]).expanduser().resolve()
FROZEN_ASSIGNMENT_MANIFEST = json.loads(
    (FROZEN_ASSIGNMENT_ROOT / "manifest.json").read_text(encoding="utf-8")
)
display(pd.DataFrame([
    {
        "method": method,
        "configured_clusters": spec["configured_clusters"],
        "observed_clusters": spec["observed_clusters"],
        "source_csv_sha256": spec["source_csv_sha256"],
    }
    for method, spec in FROZEN_ASSIGNMENT_MANIFEST["methods"].items()
]))


## 3. Load source data and define the reproducible analytical replay


In [ ]:
# Source loading, frozen-assignment validation, and LLM annotation implementation.

#!/usr/bin/env python3.12
"""Source-only runtime for the canonical final figure notebooks.

The runtime reads the public H5AD, tracked source-traced V3 clustering
assignments, and raw LLM API bundles. Assignments are hash- and exact-key
validated before embeddings, metrics, and model labels are held in memory for
one notebook process.
"""

import gzip
import hashlib
import json
import os
import re
import shutil
import sys
import tempfile
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Literal, Mapping
from urllib import request as urllib_request

import numpy as np
import pandas as pd


ROOT = REPO_ROOT
SRC_ROOT = ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DEFAULT_H5AD = ROOT / "data" / "raw" / "duke_research_repository" / "20260130_HuBMAP_experted_annotated.h5ad"
EXPECTED_H5AD_SHA256 = "5d0a59d1e7866dee5a3a06772c3c80ce7328ba6420bc140708be5ec451b8a49"
DEFAULT_ASSIGNMENTS_ROOT = ROOT / "data" / "frozen" / "v3_k300_assignments"
DEFAULT_LLM_ROOT = ROOT / "data" / "llm_api"
PLACEHOLDER_API_KEY = "REPLACE_WITH_OPENROUTER_API_KEY"

from llm_spatial_omics_clustering.final_figures_runtime.hubmap import B004_FILE_IDS

COHORT_FILE_IDS = B004_FILE_IDS

METHODS = ("leiden", "flowsom", "spatialsort", "pixie_tiff")
MODELS = ("gpt", "claude", "gemini", "deepseek")
MODEL_IDS = {
    "gpt": "openai/gpt-5.6-sol",
    "claude": "anthropic/claude-opus-5",
    "gemini": "google/gemini-3.1-pro-preview",
    "deepseek": "deepseek/deepseek-v4-pro",
}
METHOD_LABELS = {"leiden": "Leiden", "flowsom": "FlowSOM", "spatialsort": "SpatialSort", "pixie_tiff": "TIFF PIXIE"}
MODEL_LABELS = {"gpt": "GPT", "claude": "Claude", "gemini": "Gemini", "deepseek": "DeepSeek"}

TRUTH_MAP = {
    "Epithelial": "Enterocyte",
    "CD66+ Epithelial": "CD66+ Enterocyte",
    "MUC1+ Epithelial": "MUC1+ Enterocyte",
    "CD57+ Epithelial": "Enterocyte",
    "ITLN+ Epithelial": "Goblet",
    "TA": "Cycling TA",
    "CD4+ T": "CD4+ T cell",
    "Stromal": "Stroma",
    "PDPN+ Stromal": "Lymphatic",
    "CD36 high Endothelial": "Endothelial",
    "CD36 low Endothelial": "Endothelial",
    "CD49a+ Smooth muscle": "Smooth muscle",
    "Paneth": "Neuroendocrine",
    "M1 Macrophage": "M2 Macrophage",
    "NK": "CD7+ Immune",
}
ONTOLOGY = (
    "B", "CD4+ T cell", "CD66+ Enterocyte", "CD7+ Immune", "CD8+ T", "Cycling TA", "DC",
    "Endothelial", "Enterocyte", "Goblet", "ICC", "Lymphatic", "M2 Macrophage",
    "MUC1+ Enterocyte", "Nerve", "Neuroendocrine", "Neutrophil", "Noise", "Plasma",
    "Smooth muscle", "Stroma",
)


class SourceRebuildError(RuntimeError):
    """Raised when a source-only rebuild cannot satisfy its input contract."""


@dataclass(frozen=True)
class SourceInputs:
    h5ad: Path
    assignments_root: Path
    llm_data_root: Path
    llm_mode: Literal["cached", "live"] = "cached"
    api_keys: Mapping[str, str] = field(default_factory=lambda: {"openrouter": PLACEHOLDER_API_KEY})
    live_output_root: Path | None = None

    @classmethod
    def from_environment(cls) -> "SourceInputs":
        mode = os.environ.get("SOURCE_REBUILD_LLM_MODE", "live").strip().lower()
        if mode not in {"cached", "live"}:
            raise SourceRebuildError("SOURCE_REBUILD_LLM_MODE must be 'cached' or 'live'")
        return cls(
            h5ad=Path(os.environ.get("SOURCE_REBUILD_H5AD_PATH", str(DEFAULT_H5AD))).expanduser().resolve(),
            assignments_root=Path(os.environ.get("SOURCE_REBUILD_ASSIGNMENTS_ROOT", str(DEFAULT_ASSIGNMENTS_ROOT))).expanduser().resolve(),
            llm_data_root=Path(os.environ.get("SOURCE_REBUILD_LLM_DATA_ROOT", str(DEFAULT_LLM_ROOT))).expanduser().resolve(),
            llm_mode=mode,  # type: ignore[arg-type]
            api_keys={"openrouter": os.environ.get("OPENROUTER_API_KEY", PLACEHOLDER_API_KEY)},
            live_output_root=(Path(os.environ["SOURCE_REBUILD_LLM_OUTPUT_ROOT"]).expanduser().resolve() if os.environ.get("SOURCE_REBUILD_LLM_OUTPUT_ROOT") else None),
        )

def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _require_file(path: Path, label: str) -> Path:
    path = path.expanduser().resolve()
    if not path.is_file():
        raise SourceRebuildError(f"{label} is missing: {path}")
    return path


def _canonical_cluster(value: object) -> str:
    text = str(value).strip()
    match = re.search(r"(?:cluster|target|group)[_:\- ]*([0-9]+)$", text, re.IGNORECASE)
    if match:
        return str(int(match.group(1)))
    return str(int(text)) if text.isdigit() else text


def _infer_method(path: Path, payload: Mapping[str, Any]) -> str | None:
    provenance = payload.get("provenance")
    values = [payload.get("call_id"), path.name]
    if isinstance(provenance, Mapping):
        values.append(provenance.get("request_metadata", {}).get("call_id"))
    text = " ".join(str(value).lower() for value in values if value)
    return next((method for method in METHODS if method in text), None)


def _infer_model(path: Path, payload: Mapping[str, Any]) -> str | None:
    provenance = payload.get("provenance")
    values = [payload.get("model_key"), path.name]
    if isinstance(provenance, Mapping):
        values.append(provenance.get("model_key"))
    text = " ".join(str(value).lower() for value in values if value)
    return next((model for model in MODELS if model in text), None)


def _annotation_pairs(payload: Mapping[str, Any]) -> list[tuple[str, str]]:
    annotations = payload.get("annotations")
    if isinstance(annotations, Mapping):
        items = annotations.items()
    elif isinstance(annotations, list):
        items = ((item.get("target_id"), item.get("label")) for item in annotations if isinstance(item, Mapping))
    else:
        items = ()
    pairs: list[tuple[str, str]] = []
    for target, value in items:
        if isinstance(value, Mapping):
            value = value.get("label") or value.get("predicted_label") or value.get("annotation")
        if target is None or value is None:
            continue
        label = str(value).strip()
        if label not in ONTOLOGY:
            raise SourceRebuildError(f"LLM response contains a label outside the ontology: {label!r}")
        pairs.append((_canonical_cluster(target), label))
    return pairs


def _json_files(root: Path, directory_name: str) -> list[Path]:
    if not root.is_dir():
        raise SourceRebuildError(f"LLM API data root is missing: {root}")
    return sorted(path for path in root.rglob("*.json") if directory_name in path.parts)


class SourceContext:
    """Lazy source data, clustering, annotation, and metric state."""

    def __init__(self, inputs: SourceInputs):
        self.inputs = inputs
        self._scratch = Path(tempfile.mkdtemp(prefix="cell_masks_final_source_rebuild_"))
        self._prepared = False
        self._cells: pd.DataFrame | None = None
        self._features: np.ndarray | None = None
        self._markers: tuple[str, ...] | None = None
        self._assignments: dict[str, pd.DataFrame] = {}
        self._predictions: dict[tuple[str, str], pd.DataFrame] = {}
        self._embedding: np.ndarray | None = None

    def close(self) -> None:
        shutil.rmtree(self._scratch, ignore_errors=True)

    def status(self) -> dict[str, object]:
        return {
            "runtime_boundary": "H5AD + tracked frozen K=300 assignments + OpenRouter key (or cached raw LLM API bundles)",
            "h5ad": str(self.inputs.h5ad),
            "h5ad_exists": self.inputs.h5ad.is_file(),
            "assignments_root": str(self.inputs.assignments_root),
            "assignment_manifest_present": (self.inputs.assignments_root / "manifest.json").is_file(),
            "llm_data_root": str(self.inputs.llm_data_root),
            "llm_mode": self.inputs.llm_mode,
            "api_key_persisted": False,
            "prepared": self._prepared,
        }

    @property
    def cells(self) -> pd.DataFrame:
        self.prepare()
        assert self._cells is not None
        return self._cells

    @property
    def features(self) -> np.ndarray:
        self.prepare()
        assert self._features is not None
        return self._features

    @property
    def markers(self) -> tuple[str, ...]:
        self.prepare()
        assert self._markers is not None
        return self._markers

    @property
    def assignments(self) -> Mapping[str, pd.DataFrame]:
        self.prepare()
        return self._assignments

    @property
    def predictions(self) -> Mapping[tuple[str, str], pd.DataFrame]:
        self.prepare()
        return self._predictions

    def prepare(self) -> None:
        if self._prepared:
            return
        self._load_h5ad()
        self._load_frozen_assignments()
        self._predictions = self._load_cached_predictions() if self.inputs.llm_mode == "cached" else self._generate_live_predictions()
        self._prepared = True

    def _load_h5ad(self) -> None:
        try:
            import anndata as ad
            from scipy import sparse
        except ImportError as exc:  # pragma: no cover
            raise SourceRebuildError("H5AD rebuilding requires anndata and scipy") from exc
        h5ad_path = _require_file(self.inputs.h5ad, "H5AD source")
        observed_h5ad_sha256 = _sha256(h5ad_path)
        if observed_h5ad_sha256 != EXPECTED_H5AD_SHA256:
            raise SourceRebuildError(
                f"H5AD source SHA-256 {observed_h5ad_sha256}; expected {EXPECTED_H5AD_SHA256}"
            )
        dataset = ad.read_h5ad(h5ad_path, backed="r")
        try:
            required = {"File_ID", "ID", "x", "y", "cell_type_update"}
            missing = sorted(required.difference(dataset.obs.columns))
            if missing:
                raise SourceRebuildError(f"H5AD is missing required obs columns: {missing}")
            file_ids = dataset.obs["File_ID"].astype(str)
            positions = np.flatnonzero(file_ids.isin(COHORT_FILE_IDS).to_numpy())
            obs = dataset.obs.iloc[positions][["File_ID", "ID", "x", "y", "cell_type_update"]].copy()
            obs["File_ID"] = obs["File_ID"].astype(str)
            obs["ID"] = pd.to_numeric(obs["ID"], errors="raise").astype("int64")
            obs["x"] = pd.to_numeric(obs["x"], errors="raise").astype(float)
            obs["y"] = pd.to_numeric(obs["y"], errors="raise").astype(float)
            if obs.duplicated(["File_ID", "ID"]).any():
                raise SourceRebuildError("H5AD cohort contains duplicate exact keys")
            values = dataset.X[positions]
            if sparse.issparse(values):
                values = values.toarray()
            values = np.asarray(values, dtype=np.float32)
            markers = tuple(str(value) for value in dataset.var_names)
        finally:
            dataset.file.close()
        if len(obs) != 220_082:
            raise SourceRebuildError(f"H5AD cohort contains {len(obs):,} rows; expected 220,082")
        truth = obs["cell_type_update"].astype(str).replace(TRUTH_MAP)
        invalid = sorted(set(truth) - set(ONTOLOGY))
        if invalid:
            raise SourceRebuildError(f"H5AD truth labels are outside the ontology: {invalid}")
        obs["truth_label"] = truth.to_numpy()
        self._cells, self._features, self._markers = obs.reset_index(drop=True), values, markers

    def _load_frozen_assignments(self) -> None:
        """Load and validate the exact source-traced V3 publication partitions."""
        assert self._cells is not None
        root = self.inputs.assignments_root
        manifest_path = _require_file(root / "manifest.json", "frozen assignment manifest")
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("schema_version") != "llm_spatial_omics_clustering.frozen_v3_k300_assignments.v1":
            raise SourceRebuildError(f"Unexpected frozen assignment manifest schema: {manifest_path}")
        cohort = manifest.get("cohort", {})
        if int(cohort.get("cells", -1)) != len(self._cells):
            raise SourceRebuildError("Frozen assignment manifest cell count differs from the H5AD cohort")
        specs = manifest.get("methods")
        if not isinstance(specs, Mapping):
            raise SourceRebuildError("Frozen assignment manifest has no method contracts")
        keys = self._cells[["File_ID", "ID"]].copy()
        method_manifest_keys = {"leiden": "leiden", "flowsom": "flowsom", "spatialsort": "spatialsort", "pixie_tiff": "pixie"}
        for method in METHODS:
            spec = specs.get(method_manifest_keys[method])
            if not isinstance(spec, Mapping):
                raise SourceRebuildError(f"Frozen assignment manifest is missing {method}")
            path = _require_file(root / str(spec["path"]), f"{method} frozen assignments")
            if _sha256(path) != str(spec["gzip_sha256"]):
                raise SourceRebuildError(f"{method} gzip SHA-256 differs from the frozen manifest")
            digest = hashlib.sha256()
            with gzip.open(path, "rb") as handle:
                for block in iter(lambda: handle.read(1024 * 1024), b""):
                    digest.update(block)
            if digest.hexdigest() != str(spec["source_csv_sha256"]):
                raise SourceRebuildError(f"{method} decompressed CSV SHA-256 differs from the frozen manifest")
            label_column = str(spec["assignment_column"])
            frame = pd.read_csv(path, usecols=["File_ID", "ID", label_column])
            frame["File_ID"] = frame["File_ID"].astype(str)
            frame["ID"] = pd.to_numeric(frame["ID"], errors="raise").astype("int64")
            if len(frame) != len(keys) or frame.duplicated(["File_ID", "ID"]).any():
                raise SourceRebuildError(f"{method} frozen assignments violate the exact-key row contract")
            joined = keys.merge(frame, on=["File_ID", "ID"], how="left", validate="one_to_one", sort=False)
            if joined[label_column].isna().any():
                raise SourceRebuildError(f"{method} frozen assignments do not cover every H5AD cell")
            joined["cluster"] = joined[label_column].map(_canonical_cluster)
            observed = int(joined["cluster"].nunique())
            if observed != int(spec["observed_clusters"]):
                raise SourceRebuildError(
                    f"{method} has {observed} occupied clusters; expected {spec['observed_clusters']}"
                )
            if int(spec["configured_clusters"]) != 300:
                raise SourceRebuildError(f"{method} is not bound to the K=300 publication contract")
            self._assignments[method] = joined[["File_ID", "ID", "cluster"]]

    def _load_cached_predictions(self) -> dict[tuple[str, str], pd.DataFrame]:
        responses = _json_files(self.inputs.llm_data_root, "responses")
        if not responses:
            raise SourceRebuildError(f"No raw LLM response bundles found under {self.inputs.llm_data_root}")
        labels: dict[tuple[str, str], dict[str, str]] = {}
        for path in responses:
            payload = json.loads(path.read_text(encoding="utf-8"))
            if not isinstance(payload, Mapping):
                continue
            method, model = _infer_method(path, payload), _infer_model(path, payload)
            if method not in METHODS or model not in MODELS:
                continue
            target = labels.setdefault((method, model), {})
            for cluster, label in _annotation_pairs(payload):
                if cluster in target and target[cluster] != label:
                    raise SourceRebuildError(f"Conflicting cached labels for {method}/{model}/{cluster}")
                target[cluster] = label
        expected = {(method, model) for method in METHODS for model in MODELS}
        missing_sources = sorted(expected - set(labels))
        if missing_sources:
            raise SourceRebuildError(f"Raw LLM API data lacks method/model sources: {missing_sources}")
        predictions: dict[tuple[str, str], pd.DataFrame] = {}
        for method in METHODS:
            assignment = self._assignments[method]
            for model in MODELS:
                label_map = labels[(method, model)]
                missing = sorted(set(assignment["cluster"]) - set(label_map))
                if missing:
                    raise SourceRebuildError(f"Raw LLM data lacks {len(missing)} {method}/{model} cluster labels")
                frame = assignment[["File_ID", "ID"]].copy()
                frame["predicted_label"] = assignment["cluster"].map(label_map).to_numpy()
                predictions[(method, model)] = frame
        return predictions

    def _cluster_profiles(self, method: str) -> list[dict[str, Any]]:
        assert self._features is not None and self._markers is not None and self._cells is not None
        assignment = self._assignments[method]
        transformed = np.arcsinh(self._features / 5.0)
        rows: list[dict[str, Any]] = []
        for cluster, indices in assignment.groupby("cluster", sort=True).groups.items():
            values = transformed[np.asarray(indices, dtype=np.int64)]
            mean = values.mean(axis=0)
            order = np.argsort(mean)
            rows.append({"cluster": str(cluster), "n_cells": int(len(indices)), "top_markers": [self.markers[int(i)] for i in order[-12:][::-1]], "low_markers": [self.markers[int(i)] for i in order[:8]]})
        return rows

    def _generate_live_predictions(self) -> dict[tuple[str, str], pd.DataFrame]:
        key = str(self.inputs.api_keys.get("openrouter", PLACEHOLDER_API_KEY))
        if key == PLACEHOLDER_API_KEY or not key.strip():
            raise SourceRebuildError("Live mode requires OPENROUTER_API_KEY; the notebook contains only a placeholder")
        predictions: dict[tuple[str, str], pd.DataFrame] = {}
        for method in METHODS:
            profiles = self._cluster_profiles(method)
            for model in MODELS:
                labels: dict[str, str] = {}
                for start in range(0, len(profiles), 20):
                    batch = profiles[start : start + 20]
                    payload = {
                        "model": MODEL_IDS[model],
                        "temperature": 0,
                        "messages": [
                            {"role": "system", "content": "Return only a JSON object mapping cluster IDs to allowed labels."},
                            {"role": "user", "content": json.dumps({"method": method, "allowed_labels": list(ONTOLOGY), "targets": batch}, sort_keys=True)},
                        ],
                    }
                    raw = self._openrouter_request(key, payload)
                    content = raw["choices"][0]["message"]["content"]
                    parsed = json.loads(content) if isinstance(content, str) else content
                    if not isinstance(parsed, Mapping):
                        raise SourceRebuildError(f"Live {method}/{model} response was not a JSON object")
                    for cluster, label in parsed.items():
                        if str(label) not in ONTOLOGY:
                            raise SourceRebuildError(f"Live response used an invalid ontology label: {label}")
                        labels[_canonical_cluster(cluster)] = str(label)
                    if self.inputs.live_output_root is not None:
                        output = self.inputs.live_output_root / method / model / f"batch_{start // 20:03d}.json"
                        output.parent.mkdir(parents=True, exist_ok=True)
                        output.write_text(json.dumps({"method": method, "model": model, "request": payload, "raw_response": raw, "annotations": parsed}, indent=2) + "\n", encoding="utf-8")
                assignment = self._assignments[method]
                missing = sorted(set(assignment["cluster"]) - set(labels))
                if missing:
                    raise SourceRebuildError(f"Live {method}/{model} response missed clusters: {missing[:5]}")
                frame = assignment[["File_ID", "ID"]].copy()
                frame["predicted_label"] = assignment["cluster"].map(labels).to_numpy()
                predictions[(method, model)] = frame
        return predictions

    @staticmethod
    def _openrouter_request(api_key: str, payload: Mapping[str, Any]) -> Mapping[str, Any]:
        request = urllib_request.Request("https://openrouter.ai/api/v1/chat/completions", data=json.dumps(payload).encode("utf-8"), headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json", "HTTP-Referer": "https://localhost/", "X-Title": "Cell masks final source rebuild"}, method="POST")
        with urllib_request.urlopen(request, timeout=300) as response:
            result = json.loads(response.read().decode("utf-8"))
        if not isinstance(result, Mapping):
            raise SourceRebuildError("OpenRouter returned a non-object response")
        return result

    def embedding(self) -> np.ndarray:
        self.prepare()
        if self._embedding is None:
            from sklearn.decomposition import PCA

            self._embedding = PCA(n_components=2, random_state=42, svd_solver="randomized").fit_transform(np.arcsinh(self.features / 5.0))
        return self._embedding

    def metric_frame(self, method: str, model: str) -> pd.DataFrame:
        frame = self.cells[["File_ID", "ID", "truth_label"]].merge(self.predictions[(method, model)], on=["File_ID", "ID"], validate="one_to_one")
        frame["correct"] = frame["truth_label"].eq(frame["predicted_label"])
        frame["cluster"] = self._assignments[method]["cluster"].to_numpy()
        return frame

    def accuracy_matrix(self) -> pd.DataFrame:
        return pd.DataFrame({model: [float(self.metric_frame(method, model)["correct"].mean()) for method in METHODS] for model in MODELS}, index=METHODS)

    def truth_label_order(self) -> list[str]:
        return list(self.cells["truth_label"].value_counts().index.astype(str))


def source_context_from_environment() -> SourceContext:
    return SourceContext(SourceInputs.from_environment())


def cluster_configuration_frame() -> pd.DataFrame:
    return pd.DataFrame([
        {"Method": "Leiden", "Configured clusters": 300, "Observed clusters": 300, "Settings": "47 markers; asinh/5; per-FOV scaling; K320 hierarchy coarsened to K300; seed 42"},
        {"Method": "FlowSOM", "Configured clusters": 300, "Observed clusters": 300, "Settings": "45 markers; asinh/5; robust scaling; MiniSOM 32x32; Ward K300; seed 42"},
        {"Method": "SpatialSort", "Configured clusters": 300, "Observed clusters": 246, "Settings": "45 markers; neighbors 24; precision 0.65; trace 8; DMH 1; seed 42"},
        {"Method": "TIFF PIXIE", "Configured clusters": 300, "Observed clusters": 300, "Settings": "48-channel TIFF; pixel 10x10/20; cell 24x24/300; seed 42"},
    ])


def runtime_manifest() -> dict[str, Any]:
    return {
        "schema_version": "cell_masks.final_source_rebuild_runtime.v1",
        "runtime_dependency_boundary": "H5AD + tracked frozen K=300 assignments + OpenRouter key (or cached raw LLM API bundles)",
        "runtime_helper": "embedded notebook source using src/llm_spatial_omics_clustering/final_figures_runtime/",
        "panel_renderer": "embedded notebook panel cells",
        "allowed_input_environment": ["SOURCE_REBUILD_REPO_ROOT", "SOURCE_REBUILD_H5AD_PATH", "SOURCE_REBUILD_ASSIGNMENTS_ROOT", "SOURCE_REBUILD_LLM_DATA_ROOT", "SOURCE_REBUILD_LLM_MODE", "SOURCE_REBUILD_OUTPUT_ROOT", "SOURCE_REBUILD_LLM_OUTPUT_ROOT"],
        "api_key_environment": "OPENROUTER_API_KEY",
        "api_key_policy": "placeholder in notebook; environment-only in live mode; never persisted",
        "cohort_file_ids": list(COHORT_FILE_IDS),
        "methods": list(METHODS),
        "models": list(MODELS),
    }


## 4. Load frozen clustering assignments and LLM annotations


In [ ]:
# Build all source-derived state once for this notebook process.
# This loads the H5AD, validates the exact frozen K=300 assignments,
# calls OpenRouter in live mode, and retains assignments/annotations only in memory.
SOURCE_INPUTS = SourceInputs.from_environment()
SOURCE_CONTEXT = SourceContext(SOURCE_INPUTS)
SOURCE_CONTEXT.prepare()
PANEL_FIGURES = {}
display(pd.DataFrame([SOURCE_CONTEXT.status()]))


## 5. Render publication panels


### Panel A

vector workflow rebuilt in Matplotlib.


In [ ]:
# Panel A: vector workflow rebuilt in Matplotlib

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

import matplotlib.pyplot as plt

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def _workflow_box(ax, xy, width, height, text, facecolor, *, fontsize=9):
    box = FancyBboxPatch(
        xy,
        width,
        height,
        boxstyle="round,pad=0.02,rounding_size=0.04",
        linewidth=1.2,
        edgecolor="#1f2933",
        facecolor=facecolor,
    )
    ax.add_patch(box)
    ax.text(
        xy[0] + width / 2,
        xy[1] + height / 2,
        text,
        ha="center",
        va="center",
        fontsize=fontsize,
        color="#101820",
        wrap=True,
    )

def _workflow_arrow(ax, start, end):
    ax.add_patch(
        FancyArrowPatch(
            start,
            end,
            arrowstyle="-|>",
            mutation_scale=12,
            linewidth=1.2,
            color="#52606d",
            connectionstyle="arc3,rad=0.0",
        )
    )

def render_figure_02_panel_a(context):
    """Rebuild the Figure 2A clustering benchmark workflow as vector art."""
    del context
    fig, ax = plt.subplots(figsize=(12, 4.6), constrained_layout=True)
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 4.6)
    ax.axis("off")
    ax.text(0.05, 4.25, "A", fontsize=16, fontweight="bold", va="center")
    ax.text(0.55, 4.25, "Clustering method benchmark", fontsize=14, fontweight="bold", va="center")

    boxes = [
        ((0.45, 2.3), 1.6, 1.0, "H5AD\ncell features", "#d9eef7"),
        ((2.7, 2.3), 1.8, 1.0, "Eight-FOV\ncohort", "#d9eef7"),
        ((5.2, 3.15), 1.9, 0.82, "Leiden", "#f9e3b8"),
        ((5.2, 2.12), 1.9, 0.82, "FlowSOM", "#f9e3b8"),
        ((5.2, 1.09), 1.9, 0.82, "SpatialSort", "#f9e3b8"),
        ((8.05, 2.3), 1.9, 1.0, "Cluster labels\nper cell", "#d9eef7"),
        ((10.35, 2.3), 1.25, 1.0, "Purity\nand F1", "#cdebd6"),
    ]
    for xy, width, height, label, color in boxes:
        _workflow_box(ax, xy, width, height, label, color)
    _workflow_arrow(ax, (2.05, 2.8), (2.7, 2.8))
    _workflow_arrow(ax, (4.5, 2.8), (5.2, 3.55))
    _workflow_arrow(ax, (4.5, 2.8), (5.2, 2.53))
    _workflow_arrow(ax, (4.5, 2.8), (5.2, 1.5))
    _workflow_arrow(ax, (7.1, 3.55), (8.05, 2.8))
    _workflow_arrow(ax, (7.1, 2.53), (8.05, 2.8))
    _workflow_arrow(ax, (7.1, 1.5), (8.05, 2.8))
    _workflow_arrow(ax, (9.95, 2.8), (10.35, 2.8))
    ax.text(6.1, 0.35, "All assignments, labels, and metrics are recomputed in memory from the source cohort", ha="center", fontsize=9, color="#52606d")
    return fig

FIGURE = render_figure_02_panel_a(SOURCE_CONTEXT)
PANEL_FIGURES['A'] = FIGURE
display(FIGURE)

### Panel B

H5AD plus frozen source-traced assignments and Matplotlib rendering.


In [ ]:
# Panel B: H5AD plus frozen source-traced assignments and Matplotlib rendering

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

from collections.abc import Mapping

from typing import Any

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

def _prepare(context: SourceContext) -> SourceContext:
    context.prepare()
    return context

def _truth(context: SourceContext) -> pd.Series:
    return _prepare(context).cells["truth_label"].astype(str).reset_index(drop=True)

def _add_panel_label(figure: Any, panel_id: str) -> None:
    figure.text(0.01, 0.99, panel_id, ha="left", va="top", fontsize=18, fontweight="bold")

def render_figure_02_panel_b(context: SourceContext) -> Any:
    counts = _truth(context).value_counts().sort_values(ascending=False)
    figure, axis = plt.subplots(figsize=(8.5, 4.6))
    axis.bar(np.arange(len(counts)), counts.to_numpy(), color="#3e7c9a")
    axis.set(xticks=np.arange(len(counts)), xticklabels=counts.index, ylabel="Cells", title="Source cohort cell-type distribution")
    axis.tick_params(axis="x", rotation=75)
    _add_panel_label(figure, "B")
    figure.tight_layout()
    return figure

FIGURE = render_figure_02_panel_b(SOURCE_CONTEXT)
PANEL_FIGURES['B'] = FIGURE
display(FIGURE)

### Panel C

H5AD plus frozen source-traced assignments and Matplotlib rendering.


In [ ]:
# Panel C: H5AD plus frozen source-traced assignments and Matplotlib rendering

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

from collections.abc import Mapping

from typing import Any

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

def _prepare(context: SourceContext) -> SourceContext:
    context.prepare()
    return context

def _truth(context: SourceContext) -> pd.Series:
    return _prepare(context).cells["truth_label"].astype(str).reset_index(drop=True)

def _palette(values: pd.Series) -> tuple[np.ndarray, Any]:
    labels = pd.Index(values.astype(str).unique()).sort_values()
    categories = pd.Categorical(values.astype(str), categories=labels, ordered=True)
    return categories.codes, plt.get_cmap("turbo", max(1, len(labels)))

def _add_panel_label(figure: Any, panel_id: str) -> None:
    figure.text(0.01, 0.99, panel_id, ha="left", va="top", fontsize=18, fontweight="bold")

def _representative_fov(context: SourceContext) -> str:
    counts = _prepare(context).cells["File_ID"].astype(str).value_counts()
    return str(counts.sort_values(ascending=False).index[0])

def _spatial_scatter(axis: Any, context: SourceContext, labels: pd.Series, title: str) -> None:
    cells = _prepare(context).cells
    fov = _representative_fov(context)
    mask = cells["File_ID"].astype(str).eq(fov).to_numpy()
    codes, cmap = _palette(labels.loc[mask].reset_index(drop=True))
    axis.scatter(cells.loc[mask, "x"], cells.loc[mask, "y"], c=codes, cmap=cmap, s=0.45, linewidths=0, rasterized=True)
    axis.set(title=title, xticks=[], yticks=[], aspect="equal")
    axis.invert_yaxis()

def render_figure_02_panel_c(context: SourceContext) -> Any:
    figure, axis = plt.subplots(figsize=(5.6, 5.2))
    _spatial_scatter(axis, context, _truth(context), "Ground-truth cell types in representative FOV")
    _add_panel_label(figure, "C")
    figure.tight_layout()
    return figure

FIGURE = render_figure_02_panel_c(SOURCE_CONTEXT)
PANEL_FIGURES['C'] = FIGURE
display(FIGURE)

### Panel D

H5AD plus frozen source-traced assignments and Matplotlib rendering.


In [ ]:
# Panel D: H5AD plus frozen source-traced assignments and Matplotlib rendering

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

from collections.abc import Mapping

from typing import Any

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

def _prepare(context: SourceContext) -> SourceContext:
    context.prepare()
    return context

def _keys(context: SourceContext) -> pd.DataFrame:
    return _prepare(context).cells.loc[:, ["File_ID", "ID"]].copy()

def _truth(context: SourceContext) -> pd.Series:
    return _prepare(context).cells["truth_label"].astype(str).reset_index(drop=True)

def _assignment(context: SourceContext, method: str) -> pd.Series:
    if method not in METHODS:
        raise SourceRebuildError(f"Unknown clustering method: {method}")
    frame = _prepare(context).assignments[method].loc[:, ["File_ID", "ID", "cluster"]]
    merged = _keys(context).merge(frame, on=["File_ID", "ID"], how="left", validate="one_to_one", sort=False)
    if merged["cluster"].isna().any():
        raise SourceRebuildError(f"{method} did not return an assignment for every source cell")
    return merged["cluster"].astype(str).reset_index(drop=True)

def _palette(values: pd.Series) -> tuple[np.ndarray, Any]:
    labels = pd.Index(values.astype(str).unique()).sort_values()
    categories = pd.Categorical(values.astype(str), categories=labels, ordered=True)
    return categories.codes, plt.get_cmap("turbo", max(1, len(labels)))

def _add_panel_label(figure: Any, panel_id: str) -> None:
    figure.text(0.01, 0.99, panel_id, ha="left", va="top", fontsize=18, fontweight="bold")

def _umap_scatter(axis: Any, context: SourceContext, labels: pd.Series, title: str) -> None:
    coordinates = _prepare(context).embedding()
    codes, cmap = _palette(labels)
    axis.scatter(coordinates[:, 0], coordinates[:, 1], c=codes, cmap=cmap, s=0.28, linewidths=0, rasterized=True)
    axis.set(title=title, xticks=[], yticks=[], xlabel="UMAP 1", ylabel="UMAP 2")

def render_figure_02_panel_d(context: SourceContext) -> Any:
    figure, axes = plt.subplots(1, 5, figsize=(17.0, 3.7), constrained_layout=True)
    _umap_scatter(axes[0], context, _truth(context), "Truth")
    for axis, method in zip(axes[1:], METHODS, strict=True):
        _umap_scatter(axis, context, _assignment(context, method), METHOD_LABELS[method])
    _add_panel_label(figure, "D")
    return figure

FIGURE = render_figure_02_panel_d(SOURCE_CONTEXT)
PANEL_FIGURES['D'] = FIGURE
display(FIGURE)

### Panel E

H5AD plus frozen source-traced assignments and Matplotlib rendering.


In [ ]:
# Panel E: H5AD plus frozen source-traced assignments and Matplotlib rendering

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

from collections.abc import Mapping

from typing import Any

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

def _prepare(context: SourceContext) -> SourceContext:
    context.prepare()
    return context

def _keys(context: SourceContext) -> pd.DataFrame:
    return _prepare(context).cells.loc[:, ["File_ID", "ID"]].copy()

def _truth(context: SourceContext) -> pd.Series:
    return _prepare(context).cells["truth_label"].astype(str).reset_index(drop=True)

def _assignment(context: SourceContext, method: str) -> pd.Series:
    if method not in METHODS:
        raise SourceRebuildError(f"Unknown clustering method: {method}")
    frame = _prepare(context).assignments[method].loc[:, ["File_ID", "ID", "cluster"]]
    merged = _keys(context).merge(frame, on=["File_ID", "ID"], how="left", validate="one_to_one", sort=False)
    if merged["cluster"].isna().any():
        raise SourceRebuildError(f"{method} did not return an assignment for every source cell")
    return merged["cluster"].astype(str).reset_index(drop=True)

def _add_panel_label(figure: Any, panel_id: str) -> None:
    figure.text(0.01, 0.99, panel_id, ha="left", va="top", fontsize=18, fontweight="bold")

def _majority_labels(truth: pd.Series, clusters: pd.Series) -> pd.Series:
    counts = pd.crosstab(clusters.astype(str), truth.astype(str))
    if counts.empty:
        raise SourceRebuildError("Cannot derive cluster-majority labels from an empty assignment")
    lookup = counts.idxmax(axis=1).to_dict()
    return clusters.astype(str).map(lookup).astype(str)

def _purity(truth: pd.Series, clusters: pd.Series) -> float:
    counts = pd.crosstab(clusters.astype(str), truth.astype(str))
    return float(counts.max(axis=1).sum() / len(truth))

def _clustering_metrics(context: SourceContext) -> pd.DataFrame:
    truth = _truth(context)
    labels = sorted(truth.unique())
    records: list[dict[str, Any]] = []
    for method in METHODS:
        clusters = _assignment(context, method)
        majority = _majority_labels(truth, clusters)
        sizes = clusters.value_counts()
        records.append(
            {
                "method": method,
                "method_label": METHOD_LABELS[method],
                "cluster_count": int(sizes.size),
                "purity": _purity(truth, clusters),
                "majority_macro_f1": float(f1_score(truth, majority, labels=labels, average="macro", zero_division=0)),
                "median_cluster_size": float(sizes.median()),
            }
        )
    return pd.DataFrame.from_records(records)

def render_figure_02_panel_e(context: SourceContext) -> Any:
    metrics = _clustering_metrics(context).set_index("method_label")
    figure, axis = plt.subplots(figsize=(7.0, 4.4))
    positions = np.arange(len(metrics))
    axis.bar(positions - 0.19, metrics["purity"], width=0.38, label="Cluster purity", color="#2c7fb8")
    axis.bar(positions + 0.19, metrics["majority_macro_f1"], width=0.38, label="Majority-label macro F1", color="#7fcdbb")
    axis.set(xticks=positions, xticklabels=metrics.index, ylim=(0, 1), ylabel="Score", title="Clustering agreement with H5AD truth labels")
    axis.legend(frameon=False)
    _add_panel_label(figure, "E")
    figure.tight_layout()
    return figure

FIGURE = render_figure_02_panel_e(SOURCE_CONTEXT)
PANEL_FIGURES['E'] = FIGURE
display(FIGURE)

### Panel F

H5AD plus frozen source-traced assignments and Matplotlib rendering.


In [ ]:
# Panel F: H5AD plus frozen source-traced assignments and Matplotlib rendering

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

from collections.abc import Mapping

from typing import Any

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

def _prepare(context: SourceContext) -> SourceContext:
    context.prepare()
    return context

def _keys(context: SourceContext) -> pd.DataFrame:
    return _prepare(context).cells.loc[:, ["File_ID", "ID"]].copy()

def _truth(context: SourceContext) -> pd.Series:
    return _prepare(context).cells["truth_label"].astype(str).reset_index(drop=True)

def _assignment(context: SourceContext, method: str) -> pd.Series:
    if method not in METHODS:
        raise SourceRebuildError(f"Unknown clustering method: {method}")
    frame = _prepare(context).assignments[method].loc[:, ["File_ID", "ID", "cluster"]]
    merged = _keys(context).merge(frame, on=["File_ID", "ID"], how="left", validate="one_to_one", sort=False)
    if merged["cluster"].isna().any():
        raise SourceRebuildError(f"{method} did not return an assignment for every source cell")
    return merged["cluster"].astype(str).reset_index(drop=True)

def _add_panel_label(figure: Any, panel_id: str) -> None:
    figure.text(0.01, 0.99, panel_id, ha="left", va="top", fontsize=18, fontweight="bold")

def _majority_labels(truth: pd.Series, clusters: pd.Series) -> pd.Series:
    counts = pd.crosstab(clusters.astype(str), truth.astype(str))
    if counts.empty:
        raise SourceRebuildError("Cannot derive cluster-majority labels from an empty assignment")
    lookup = counts.idxmax(axis=1).to_dict()
    return clusters.astype(str).map(lookup).astype(str)

def _heatmap(axis: Any, matrix: pd.DataFrame, title: str, *, vmin: float | None = None, vmax: float | None = None, cmap: str = "viridis") -> Any:
    image = axis.imshow(matrix.to_numpy(dtype=float), aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    axis.set(
        title=title,
        xticks=np.arange(matrix.shape[1]),
        xticklabels=list(matrix.columns),
        yticks=np.arange(matrix.shape[0]),
        yticklabels=list(matrix.index),
    )
    axis.tick_params(axis="x", rotation=45)
    if matrix.shape[0] * matrix.shape[1] <= 64:
        for row_index in range(matrix.shape[0]):
            for column_index in range(matrix.shape[1]):
                axis.text(column_index, row_index, f"{matrix.iat[row_index, column_index]:.2f}", ha="center", va="center", fontsize=7)
    return image

def render_figure_02_panel_f(context: SourceContext) -> Any:
    truth = _truth(context)
    types = truth.value_counts().head(12).index.astype(str)
    matrix: dict[str, pd.Series] = {}
    for method in METHODS:
        majority = _majority_labels(truth, _assignment(context, method))
        matrix[METHOD_LABELS[method]] = pd.Series(
            {label: f1_score(truth.eq(label), majority.eq(label), zero_division=0) for label in types}
        )
    frame = pd.DataFrame(matrix).loc[types]
    figure, axis = plt.subplots(figsize=(7.5, 6.0))
    image = _heatmap(axis, frame, "Per-cell-type F1 after cluster-majority labeling", vmin=0, vmax=1)
    figure.colorbar(image, ax=axis, label="F1")
    _add_panel_label(figure, "F")
    figure.tight_layout()
    return figure

FIGURE = render_figure_02_panel_f(SOURCE_CONTEXT)
PANEL_FIGURES['F'] = FIGURE
display(FIGURE)

### Panel G

H5AD plus frozen source-traced assignments and Matplotlib rendering.


In [ ]:
# Panel G: H5AD plus frozen source-traced assignments and Matplotlib rendering

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

from collections.abc import Mapping

from typing import Any

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

def _prepare(context: SourceContext) -> SourceContext:
    context.prepare()
    return context

def _keys(context: SourceContext) -> pd.DataFrame:
    return _prepare(context).cells.loc[:, ["File_ID", "ID"]].copy()

def _assignment(context: SourceContext, method: str) -> pd.Series:
    if method not in METHODS:
        raise SourceRebuildError(f"Unknown clustering method: {method}")
    frame = _prepare(context).assignments[method].loc[:, ["File_ID", "ID", "cluster"]]
    merged = _keys(context).merge(frame, on=["File_ID", "ID"], how="left", validate="one_to_one", sort=False)
    if merged["cluster"].isna().any():
        raise SourceRebuildError(f"{method} did not return an assignment for every source cell")
    return merged["cluster"].astype(str).reset_index(drop=True)

def _add_panel_label(figure: Any, panel_id: str) -> None:
    figure.text(0.01, 0.99, panel_id, ha="left", va="top", fontsize=18, fontweight="bold")

def render_figure_02_panel_g(context: SourceContext) -> Any:
    size_vectors = []
    labels = []
    for method in METHODS:
        sizes = _assignment(context, method).value_counts().to_numpy()
        size_vectors.append(np.log10(sizes))
        labels.append(METHOD_LABELS[method])
    figure, axis = plt.subplots(figsize=(7.0, 4.6))
    axis.boxplot(size_vectors, labels=labels, showfliers=False, patch_artist=True, boxprops={"facecolor": "#c6dbef"})
    axis.set(ylabel="log10(cells per cluster)", title="Cluster-size distributions")
    axis.tick_params(axis="x", rotation=35)
    _add_panel_label(figure, "G")
    figure.tight_layout()
    return figure

FIGURE = render_figure_02_panel_g(SOURCE_CONTEXT)
PANEL_FIGURES['G'] = FIGURE
display(FIGURE)

### Panel H

H5AD plus frozen source-traced assignments and Matplotlib rendering.


In [ ]:
# Panel H: H5AD plus frozen source-traced assignments and Matplotlib rendering

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

from collections.abc import Mapping

from typing import Any

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

def _prepare(context: SourceContext) -> SourceContext:
    context.prepare()
    return context

def _keys(context: SourceContext) -> pd.DataFrame:
    return _prepare(context).cells.loc[:, ["File_ID", "ID"]].copy()

def _assignment(context: SourceContext, method: str) -> pd.Series:
    if method not in METHODS:
        raise SourceRebuildError(f"Unknown clustering method: {method}")
    frame = _prepare(context).assignments[method].loc[:, ["File_ID", "ID", "cluster"]]
    merged = _keys(context).merge(frame, on=["File_ID", "ID"], how="left", validate="one_to_one", sort=False)
    if merged["cluster"].isna().any():
        raise SourceRebuildError(f"{method} did not return an assignment for every source cell")
    return merged["cluster"].astype(str).reset_index(drop=True)

def _add_panel_label(figure: Any, panel_id: str) -> None:
    figure.text(0.01, 0.99, panel_id, ha="left", va="top", fontsize=18, fontweight="bold")

def _heatmap(axis: Any, matrix: pd.DataFrame, title: str, *, vmin: float | None = None, vmax: float | None = None, cmap: str = "viridis") -> Any:
    image = axis.imshow(matrix.to_numpy(dtype=float), aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    axis.set(
        title=title,
        xticks=np.arange(matrix.shape[1]),
        xticklabels=list(matrix.columns),
        yticks=np.arange(matrix.shape[0]),
        yticklabels=list(matrix.index),
    )
    axis.tick_params(axis="x", rotation=45)
    if matrix.shape[0] * matrix.shape[1] <= 64:
        for row_index in range(matrix.shape[0]):
            for column_index in range(matrix.shape[1]):
                axis.text(column_index, row_index, f"{matrix.iat[row_index, column_index]:.2f}", ha="center", va="center", fontsize=7)
    return image

def _marker_separation(context: SourceContext, n_markers: int = 12) -> pd.DataFrame:
    features = np.arcsinh(_prepare(context).features / 5.0)
    markers = np.asarray(context.markers, dtype=str)
    per_method: dict[str, np.ndarray] = {}
    for method in METHODS:
        clusters = _assignment(context, method).to_numpy()
        means = np.vstack([features[clusters == cluster].mean(axis=0) for cluster in pd.unique(clusters)])
        per_method[METHOD_LABELS[method]] = means.std(axis=0)
    frame = pd.DataFrame(per_method, index=markers)
    selected = frame.mean(axis=1).nlargest(n_markers).index
    return frame.loc[selected]

def render_figure_02_panel_h(context: SourceContext) -> Any:
    frame = _marker_separation(context)
    figure, axis = plt.subplots(figsize=(8.0, 5.2))
    image = _heatmap(axis, frame, "Marker separation across cluster centroids", cmap="magma")
    figure.colorbar(image, ax=axis, label="SD of cluster mean arcsinh intensity")
    _add_panel_label(figure, "H")
    figure.tight_layout()
    return figure

FIGURE = render_figure_02_panel_h(SOURCE_CONTEXT)
PANEL_FIGURES['H'] = FIGURE
display(FIGURE)

### Panel I

H5AD plus frozen source-traced assignments and Matplotlib rendering.


In [ ]:
# Panel I: H5AD plus frozen source-traced assignments and Matplotlib rendering

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

from collections.abc import Mapping

from typing import Any

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

def _prepare(context: SourceContext) -> SourceContext:
    context.prepare()
    return context

def _keys(context: SourceContext) -> pd.DataFrame:
    return _prepare(context).cells.loc[:, ["File_ID", "ID"]].copy()

def _truth(context: SourceContext) -> pd.Series:
    return _prepare(context).cells["truth_label"].astype(str).reset_index(drop=True)

def _assignment(context: SourceContext, method: str) -> pd.Series:
    if method not in METHODS:
        raise SourceRebuildError(f"Unknown clustering method: {method}")
    frame = _prepare(context).assignments[method].loc[:, ["File_ID", "ID", "cluster"]]
    merged = _keys(context).merge(frame, on=["File_ID", "ID"], how="left", validate="one_to_one", sort=False)
    if merged["cluster"].isna().any():
        raise SourceRebuildError(f"{method} did not return an assignment for every source cell")
    return merged["cluster"].astype(str).reset_index(drop=True)

def _add_panel_label(figure: Any, panel_id: str) -> None:
    figure.text(0.01, 0.99, panel_id, ha="left", va="top", fontsize=18, fontweight="bold")

def _majority_labels(truth: pd.Series, clusters: pd.Series) -> pd.Series:
    counts = pd.crosstab(clusters.astype(str), truth.astype(str))
    if counts.empty:
        raise SourceRebuildError("Cannot derive cluster-majority labels from an empty assignment")
    lookup = counts.idxmax(axis=1).to_dict()
    return clusters.astype(str).map(lookup).astype(str)

def _purity(truth: pd.Series, clusters: pd.Series) -> float:
    counts = pd.crosstab(clusters.astype(str), truth.astype(str))
    return float(counts.max(axis=1).sum() / len(truth))

def _clustering_metrics(context: SourceContext) -> pd.DataFrame:
    truth = _truth(context)
    labels = sorted(truth.unique())
    records: list[dict[str, Any]] = []
    for method in METHODS:
        clusters = _assignment(context, method)
        majority = _majority_labels(truth, clusters)
        sizes = clusters.value_counts()
        records.append(
            {
                "method": method,
                "method_label": METHOD_LABELS[method],
                "cluster_count": int(sizes.size),
                "purity": _purity(truth, clusters),
                "majority_macro_f1": float(f1_score(truth, majority, labels=labels, average="macro", zero_division=0)),
                "median_cluster_size": float(sizes.median()),
            }
        )
    return pd.DataFrame.from_records(records)

def _heatmap(axis: Any, matrix: pd.DataFrame, title: str, *, vmin: float | None = None, vmax: float | None = None, cmap: str = "viridis") -> Any:
    image = axis.imshow(matrix.to_numpy(dtype=float), aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    axis.set(
        title=title,
        xticks=np.arange(matrix.shape[1]),
        xticklabels=list(matrix.columns),
        yticks=np.arange(matrix.shape[0]),
        yticklabels=list(matrix.index),
    )
    axis.tick_params(axis="x", rotation=45)
    if matrix.shape[0] * matrix.shape[1] <= 64:
        for row_index in range(matrix.shape[0]):
            for column_index in range(matrix.shape[1]):
                axis.text(column_index, row_index, f"{matrix.iat[row_index, column_index]:.2f}", ha="center", va="center", fontsize=7)
    return image

def render_figure_02_panel_i(context: SourceContext) -> Any:
    metrics = _clustering_metrics(context).set_index("method_label")
    values = metrics[["purity", "majority_macro_f1", "cluster_count", "median_cluster_size"]].copy()
    values["cluster_count"] /= values["cluster_count"].max()
    values["median_cluster_size"] /= values["median_cluster_size"].max()
    values.columns = ["Purity", "Majority F1", "Cluster count (scaled)", "Median size (scaled)"]
    figure, axis = plt.subplots(figsize=(7.4, 4.2))
    image = _heatmap(axis, values, "Source-rebuilt clustering metric overview", vmin=0, vmax=1)
    figure.colorbar(image, ax=axis, label="Scaled score")
    _add_panel_label(figure, "I")
    figure.tight_layout()
    return figure

FIGURE = render_figure_02_panel_i(SOURCE_CONTEXT)
PANEL_FIGURES['I'] = FIGURE
display(FIGURE)

### Panel J

H5AD plus frozen source-traced assignments and Matplotlib rendering.


In [ ]:
# Panel J: H5AD plus frozen source-traced assignments and Matplotlib rendering

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

from collections.abc import Mapping

from typing import Any

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

def _prepare(context: SourceContext) -> SourceContext:
    context.prepare()
    return context

def _keys(context: SourceContext) -> pd.DataFrame:
    return _prepare(context).cells.loc[:, ["File_ID", "ID"]].copy()

def _assignment(context: SourceContext, method: str) -> pd.Series:
    if method not in METHODS:
        raise SourceRebuildError(f"Unknown clustering method: {method}")
    frame = _prepare(context).assignments[method].loc[:, ["File_ID", "ID", "cluster"]]
    merged = _keys(context).merge(frame, on=["File_ID", "ID"], how="left", validate="one_to_one", sort=False)
    if merged["cluster"].isna().any():
        raise SourceRebuildError(f"{method} did not return an assignment for every source cell")
    return merged["cluster"].astype(str).reset_index(drop=True)

def _palette(values: pd.Series) -> tuple[np.ndarray, Any]:
    labels = pd.Index(values.astype(str).unique()).sort_values()
    categories = pd.Categorical(values.astype(str), categories=labels, ordered=True)
    return categories.codes, plt.get_cmap("turbo", max(1, len(labels)))

def _add_panel_label(figure: Any, panel_id: str) -> None:
    figure.text(0.01, 0.99, panel_id, ha="left", va="top", fontsize=18, fontweight="bold")

def _representative_fov(context: SourceContext) -> str:
    counts = _prepare(context).cells["File_ID"].astype(str).value_counts()
    return str(counts.sort_values(ascending=False).index[0])

def _spatial_scatter(axis: Any, context: SourceContext, labels: pd.Series, title: str) -> None:
    cells = _prepare(context).cells
    fov = _representative_fov(context)
    mask = cells["File_ID"].astype(str).eq(fov).to_numpy()
    codes, cmap = _palette(labels.loc[mask].reset_index(drop=True))
    axis.scatter(cells.loc[mask, "x"], cells.loc[mask, "y"], c=codes, cmap=cmap, s=0.45, linewidths=0, rasterized=True)
    axis.set(title=title, xticks=[], yticks=[], aspect="equal")
    axis.invert_yaxis()

def render_figure_02_panel_j(context: SourceContext) -> Any:
    figure, axis = plt.subplots(figsize=(5.6, 5.2))
    _spatial_scatter(axis, context, _assignment(context, "pixie_tiff"), "TIFF PIXIE clusters in representative FOV")
    _add_panel_label(figure, "J")
    figure.tight_layout()
    return figure

FIGURE = render_figure_02_panel_j(SOURCE_CONTEXT)
PANEL_FIGURES['J'] = FIGURE
display(FIGURE)

## 6. Export the source-rebuilt figure PDF


In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

PDF_ROOT = Path(os.environ["SOURCE_REBUILD_OUTPUT_ROOT"]).expanduser().resolve()
PDF_ROOT.mkdir(parents=True, exist_ok=True)
PDF_PATH = PDF_ROOT / 'Figure_02_source_rebuilt.pdf'
with PdfPages(PDF_PATH) as pdf:
    for panel_id in ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']:
        PANEL_FIGURES[panel_id].savefig(pdf, format="pdf", bbox_inches="tight")
print(f"Wrote {PDF_PATH}")
